# Fraud Mitigation Agent · 00 Setup

Configura el runtime, carga dependencias y siembra únicamente datos sintéticos. La ejecución funciona sin LLM externo y puede usar memoria local si no hay `MONGODB_URI`.


In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = ""  # opcional: URL HTTPS de tu fork
REPO_DIR = "/content/fraud-mitigation-agent-workshop"
if REPO_URL and not Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymongo[srv]", "numpy", "pandas", "python-dotenv", "pytest"], check=True)
sys.path.insert(0, str(Path(REPO_DIR) / "src"))
print("Repositorio:", REPO_DIR)


In [ ]:
# En Colab, crea un Secret llamado MONGODB_URI. Nunca lo imprimas.
try:
    from google.colab import userdata
    uri = userdata.get("MONGODB_URI")
    if uri:
        os.environ["MONGODB_URI"] = uri
except Exception:
    pass

from fraud_mitigation_agent.config import Settings
from fraud_mitigation_agent.db import get_client, get_database
from fraud_mitigation_agent.local import InMemoryDB

settings = Settings.from_env()
if settings.mongodb_uri:
    client = get_client(settings.mongodb_uri)
    db = get_database(client, settings.database_name)
    print("Atlas conectado:", settings.database_name)
else:
    db = InMemoryDB()
    print("Modo local en memoria: configura MONGODB_URI para usar Atlas")


In [ ]:
from fraud_mitigation_agent.synthetic import seed_demo_data
summary = seed_demo_data(db, reset=True)
print(summary)
print("Colecciones listas: transactions, customer_state, fraud_patterns, risk_rules_config, final_outcome, risk_evidence")


In [ ]:
for tx in db.transactions.find({}, {"_id": 0, "tx_id": 1, "customer_id": 1, "ground_truth_fraud": 1}):
    print(tx)
